# ¿Cuándo caería Swift? Una predicción que mejora con cada órbita

**Categoría: predicción. Corte de los datos: 3 de noviembre de 2026 a las 21:22 UTC.**

En este notebook reconstruyo la pérdida de altura del observatorio Swift y compruebo cuánto podía aproximarme a su reentrada real, ocurrida el 4 de noviembre de 2026 a las 12:39 UTC. No mezclo el futuro con el pasado: separo entrenamiento, validación y una prueba final formada por las últimas semanas.

## Ruta de lectura

1. Cargo y ordeno el historial orbital.
2. Convierto el movimiento medio en una altura aproximada.
3. Separo los datos por fecha.
4. Comparo una recta sencilla con un modelo de potencia.
5. Mido el error en horas y reviso cómo cambia al acercarse la caída.
6. Explico qué puede y qué no puede concluirse.


## 1. Fuente y unidad de observación

Cada fila es un conjunto de elementos orbitales TLE publicado para Swift, identificado como NORAD 28485. El archivo procede del archivo histórico de CelesTrak, que redistribuye datos orbitales de USSPACECOM. Conservo el archivo original sin modificar.

La variable que quiero estimar es el número de horas que faltan desde cada observación hasta la reentrada confirmada por NASA. Este es un estudio retrospectivo de un único satélite, no un modelo general para cualquier objeto espacial.


In [1]:
import json
import math
import statistics
from datetime import datetime, timezone
from pathlib import Path

FUENTE = Path("datos/sat000028485.json")
REENTRADA = datetime(2026, 11, 4, 12, 39, tzinfo=timezone.utc)
MU_TIERRA = 398600.4418
RADIO_TIERRA = 6378.137


def fecha_utc(texto):
    return datetime.fromisoformat(texto).replace(tzinfo=timezone.utc)


def altura_media(movimiento_medio):
    movimiento_rad_s = movimiento_medio * 2 * math.pi / 86400
    return (MU_TIERRA / movimiento_rad_s**2) ** (1 / 3) - RADIO_TIERRA


def cargar_diario():
    with FUENTE.open(encoding="utf-8") as archivo:
        registros = json.load(archivo)
    por_dia = {}
    for registro in registros:
        fecha = fecha_utc(registro["EPOCH"])
        fila = dict(registro)
        fila["fecha"] = fecha
        fila["altura_media_km"] = altura_media(fila["MEAN_MOTION"])
        fila["horas_restantes"] = (REENTRADA - fecha).total_seconds() / 3600
        clave = fecha.date()
        if clave not in por_dia or fecha > por_dia[clave]["fecha"]:
            por_dia[clave] = fila
    diario = [por_dia[clave] for clave in sorted(por_dia)]
    alturas = [fila["altura_media_km"] for fila in diario]
    movimientos = [fila["MEAN_MOTION"] for fila in diario]
    for i, fila in enumerate(diario):
        for dias in (1, 3, 7, 14):
            fila[f"descenso_{dias}d_km"] = ((alturas[i - dias] - alturas[i]) / dias if i >= dias else None)
        fila["cambio_movimiento_7d"] = ((movimientos[i] - movimientos[i - 7]) / 7 if i >= 7 else None)
        fila["bstar_abs"] = abs(fila["BSTAR"])
        fila["log_margen_120"] = math.log(max(fila["altura_media_km"] - 120, 0.01))
        fila["log_descenso_3d"] = math.log(max(fila["descenso_3d_km"] or 0.0001, 0.0001))
        fila["log_descenso_7d"] = math.log(max(fila["descenso_7d_km"] or 0.0001, 0.0001))
    return registros, diario


def traspuesta(matriz):
    return [list(columna) for columna in zip(*matriz)]


def producto(a, b):
    bt = traspuesta(b)
    return [[sum(x * y for x, y in zip(fila, columna)) for columna in bt] for fila in a]


def resolver(a, b):
    n = len(b)
    ampliada = [list(a[i]) + [b[i]] for i in range(n)]
    for col in range(n):
        pivote = max(range(col, n), key=lambda fila: abs(ampliada[fila][col]))
        ampliada[col], ampliada[pivote] = ampliada[pivote], ampliada[col]
        valor = ampliada[col][col]
        if abs(valor) < 1e-12:
            raise ValueError("matriz singular")
        ampliada[col] = [x / valor for x in ampliada[col]]
        for fila in range(n):
            if fila == col:
                continue
            factor = ampliada[fila][col]
            ampliada[fila] = [x - factor * y for x, y in zip(ampliada[fila], ampliada[col])]
    return [fila[-1] for fila in ampliada]


class RegresionRidge:
    def __init__(self, alpha=0.0):
        self.alpha = alpha

    def fit(self, x, y):
        columnas = traspuesta(x)
        self.medianas = [statistics.median(v for v in col if v is not None) for col in columnas]
        limpio = [[self.medianas[j] if v is None else v for j, v in enumerate(fila)] for fila in x]
        columnas = traspuesta(limpio)
        self.medias = [statistics.fmean(col) for col in columnas]
        self.escalas = [statistics.pstdev(col) or 1 for col in columnas]
        z = [[1.0] + [(v - self.medias[j]) / self.escalas[j] for j, v in enumerate(fila)] for fila in limpio]
        zt = traspuesta(z)
        normal = producto(zt, z)
        for i in range(1, len(normal)):
            normal[i][i] += self.alpha
        objetivo = [sum(z[fila][col] * y[fila] for fila in range(len(z))) for col in range(len(z[0]))]
        self.coeficientes = resolver(normal, objetivo)
        return self

    def predict(self, x):
        predicciones = []
        for fila in x:
            limpio = [self.medianas[j] if v is None else v for j, v in enumerate(fila)]
            z = [1.0] + [(v - self.medias[j]) / self.escalas[j] for j, v in enumerate(limpio)]
            predicciones.append(sum(a * b for a, b in zip(z, self.coeficientes)))
        return predicciones


def matriz(filas, variables):
    return [[fila.get(variable) for variable in variables] for fila in filas]


def metricas(real, prediccion):
    errores = [p - r for p, r in zip(prediccion, real)]
    return {"mae_horas": statistics.fmean(abs(e) for e in errores), "rmse_horas": math.sqrt(statistics.fmean(e * e for e in errores)), "sesgo_horas": statistics.fmean(errores)}


def evaluar(nombre, modelo, variables, entrenamiento, validacion, prueba, log_objetivo=False):
    y_ent = [fila["horas_restantes"] for fila in entrenamiento]
    if log_objetivo:
        y_ent = [math.log(max(y, 0.01)) for y in y_ent]
    modelo.fit(matriz(entrenamiento, variables), y_ent)
    resultados = []
    for etiqueta, bloque in (("validación", validacion), ("prueba", prueba)):
        real = [fila["horas_restantes"] for fila in bloque]
        pred = modelo.predict(matriz(bloque, variables))
        if log_objetivo:
            pred = [math.exp(min(p, 20)) for p in pred]
        resultados.append({"modelo": nombre, "bloque": etiqueta, "n": len(bloque), **metricas(real, pred), "pred_ultima_h": pred[-1], "real_ultima_h": real[-1]})
    return resultados


def main():
    registros, diario = cargar_diario()
    corte_ent = fecha_utc("2026-09-01T00:00:00")
    corte_val = fecha_utc("2026-10-16T00:00:00")
    entrenamiento = [f for f in diario if f["fecha"] < corte_ent]
    validacion = [f for f in diario if corte_ent <= f["fecha"] < corte_val]
    prueba = [f for f in diario if f["fecha"] >= corte_val]
    base = ["altura_media_km"]
    orbitales = ["altura_media_km", "descenso_1d_km", "descenso_3d_km", "descenso_7d_km", "descenso_14d_km", "MEAN_MOTION_DOT", "bstar_abs", "cambio_movimiento_7d"]
    resultados = evaluar("referencia_altura", RegresionRidge(0), base, entrenamiento, validacion, prueba)
    resultados.extend(evaluar("referencia_potencia", RegresionRidge(0), ["log_margen_120"], entrenamiento, validacion, prueba, log_objetivo=True))
    resultados.extend(evaluar("potencia_con_descenso", RegresionRidge(1), ["log_margen_120", "log_descenso_3d", "log_descenso_7d", "bstar_abs"], entrenamiento, validacion, prueba, log_objetivo=True))
    for alpha in (0.1, 1, 10, 100, 1000):
        resultados.extend(evaluar(f"ridge_orbital_{alpha:g}", RegresionRidge(alpha), orbitales, entrenamiento, validacion, prueba))
    final = RegresionRidge(0)
    combinado = entrenamiento + validacion
    final.fit(matriz(combinado, ["log_margen_120"]), [math.log(max(f["horas_restantes"], 0.01)) for f in combinado])
    pred_final = [math.exp(p) for p in final.predict(matriz(prueba, ["log_margen_120"]))]
    real_final = [f["horas_restantes"] for f in prueba]
    resumen_final = metricas(real_final, pred_final)
    print(f"registros originales: {len(registros)}")
    print(f"días con observación: {len(diario)}")
    print(f"bloques: entrenamiento={len(entrenamiento)}, validación={len(validacion)}, prueba={len(prueba)}")
    print("\núltimas observaciones")
    for fila in diario[-10:]:
        print(fila["fecha"].isoformat(), f"altura={fila['altura_media_km']:.2f} km", f"restan={fila['horas_restantes']:.2f} h")
    print("\nresultados")
    for fila in resultados:
        print(fila["modelo"], fila["bloque"], f"n={fila['n']}", f"MAE={fila['mae_horas']:.2f} h", f"RMSE={fila['rmse_horas']:.2f} h", f"sesgo={fila['sesgo_horas']:.2f} h", f"última={fila['pred_ultima_h']:.2f}/{fila['real_ultima_h']:.2f} h")
    print("\nmodelo final reentrenado")
    print(f"MAE={resumen_final['mae_horas']:.2f} h RMSE={resumen_final['rmse_horas']:.2f} h sesgo={resumen_final['sesgo_horas']:.2f} h")
    for fila, pred in zip(prueba, pred_final):
        print(fila["fecha"].isoformat(), f"pred={pred:.2f} h", f"real={fila['horas_restantes']:.2f} h", f"error={pred-fila['horas_restantes']:.2f} h")

## 2. Preparación de los datos

Me quedo con la última observación disponible de cada día. Calculo la altura media aproximada mediante la tercera ley de Kepler y preparo velocidades de descenso de uno, tres, siete y catorce días.


In [2]:
# cargo el historial y compruebo su cobertura antes de modelar
registros, diario = cargar_diario()
print(f"Registros originales: {len(registros):,}")
print(f"Días con observación: {len(diario):,}")
print(f"Primera fecha: {diario[0]['fecha'].isoformat()}")
print(f"Última fecha: {diario[-1]['fecha'].isoformat()}")
print(f"Última altura media aproximada: {diario[-1]['altura_media_km']:.2f} km")


Registros originales: 1,594
Días con observación: 618
Primera fecha: 2025-01-01T17:07:25.478976+00:00
Última fecha: 2026-11-03T21:22:35.592672+00:00
Última altura media aproximada: 183.49 km


## 3. Separación temporal

Entreno hasta el 31 de agosto de 2026. Uso del 1 de septiembre al 15 de octubre para escoger el método y reservo desde el 16 de octubre hasta la última observación como prueba. Así simulo lo que habría sabido en cada momento.


In [3]:
# separo los periodos sin barajar las fechas
corte_ent = fecha_utc("2026-09-01T00:00:00")
corte_val = fecha_utc("2026-10-16T00:00:00")
entrenamiento = [f for f in diario if f["fecha"] < corte_ent]
validacion = [f for f in diario if corte_ent <= f["fecha"] < corte_val]
prueba = [f for f in diario if f["fecha"] >= corte_val]
print(f"Entrenamiento: {len(entrenamiento)} días")
print(f"Validación: {len(validacion)} días")
print(f"Prueba: {len(prueba)} días")


Entrenamiento: 566 días
Validación: 34 días
Prueba: 18 días


## 4. Modelo de referencia y modelo elegido

La referencia es una regresión lineal entre altura y horas restantes. Es deliberadamente sencilla. El modelo alternativo transforma ambas variables: relaciona el logaritmo del tiempo restante con el logaritmo del margen sobre 120 kilómetros. Esa forma de potencia permite representar una caída que se acelera.


In [4]:
# comparo la recta con la relación de potencia usando solo el periodo de validación
base = evaluar("recta", RegresionRidge(0), ["altura_media_km"], entrenamiento, validacion, prueba)
potencia = evaluar("potencia", RegresionRidge(0), ["log_margen_120"], entrenamiento, validacion, prueba, log_objetivo=True)
for fila in base + potencia:
    if fila["bloque"] == "validación":
        print(f"{fila['modelo']}: MAE={fila['mae_horas']:.1f} h, RMSE={fila['rmse_horas']:.1f} h")


recta: MAE=2572.2 h, RMSE=2825.1 h
potencia: MAE=428.0 h, RMSE=436.2 h


La recta produce incluso tiempos negativos al salir del rango que ha aprendido. El modelo de potencia también mantiene un error grande con semanas de antelación, pero es el único que conserva una trayectoria físicamente plausible. Lo selecciono con la validación y después lo reentreno con entrenamiento más validación, sin utilizar la prueba.


In [5]:
# reentreno el método elegido y evalúo una sola vez el tramo final
modelo = RegresionRidge(0)
combinado = entrenamiento + validacion
modelo.fit(matriz(combinado, ["log_margen_120"]), [math.log(max(f["horas_restantes"], 0.01)) for f in combinado])
prediccion = [math.exp(p) for p in modelo.predict(matriz(prueba, ["log_margen_120"]))]
real = [f["horas_restantes"] for f in prueba]
resultado = metricas(real, prediccion)
print(f"MAE de prueba: {resultado['mae_horas']:.2f} horas")
print(f"RMSE de prueba: {resultado['rmse_horas']:.2f} horas")
print(f"Sesgo: {resultado['sesgo_horas']:.2f} horas")


MAE de prueba: 32.47 horas
RMSE de prueba: 45.92 horas
Sesgo: 27.80 horas


## 5. La predicción se estrecha al final

El error medio resume el periodo, pero esconde la historia. Reviso cada actualización para comprobar cuánto faltaba realmente y cuánto estimaba el modelo.


In [6]:
# muestro la evolución sin elegir solo los aciertos más vistosos
print("fecha | predicción | realidad | error")
for fila, pred, observado in zip(prueba, prediccion, real):
    print(f"{fila['fecha'].date()} | {pred:7.1f} h | {observado:7.1f} h | {pred-observado:+7.1f} h")


fecha | predicción | realidad | error
2026-10-16 |   562.9 h |   459.2 h |  +103.7 h
2026-10-17 |   525.8 h |   435.2 h |   +90.7 h
2026-10-18 |   478.5 h |   403.7 h |   +74.8 h
2026-10-19 |   456.9 h |   388.7 h |   +68.2 h
2026-10-20 |   425.5 h |   366.2 h |   +59.2 h
2026-10-21 |   390.9 h |   342.3 h |   +48.6 h
2026-10-22 |   361.1 h |   321.3 h |   +39.8 h
2026-10-24 |   294.7 h |   273.5 h |   +21.2 h
2026-10-25 |   263.1 h |   248.2 h |   +14.9 h
2026-10-26 |   227.7 h |   216.8 h |   +10.8 h
2026-10-27 |   187.8 h |   181.1 h |    +6.7 h
2026-10-28 |   162.5 h |   158.8 h |    +3.7 h
2026-10-29 |   135.1 h |   135.0 h |    +0.1 h
2026-10-30 |   107.9 h |   109.8 h |    -1.9 h
2026-10-31 |    78.7 h |    86.1 h |    -7.3 h
2026-11-01 |    50.6 h |    62.4 h |   -11.8 h
2026-11-02 |    29.0 h |    41.7 h |   -12.7 h
2026-11-03 |     6.9 h |    15.3 h |    -8.4 h


## 6. Robustez y limitaciones

### Lo que compruebo

- La división es temporal: ninguna observación del tramo final entra en la selección del modelo.
- Comparo contra una referencia simple que fracasa de forma visible.
- Informo MAE, RMSE y sesgo, no solo el último acierto.
- Mantengo las 18 observaciones de prueba, incluidas las que se desvían más de cuatro días.

### Lo que no puedo concluir

- Solo observo una reentrada. El modelo no está validado para otros satélites.
- La altura procede de elementos TLE y es una aproximación orbital, no una medición directa continua.
- No incorporo densidad termosférica, orientación del satélite ni previsiones solares. Por eso el modelo describe la curva observada, pero no sustituye a un propagador orbital profesional.
- La reentrada real se conoce después del suceso; este ejercicio reproduce qué habría pasado al ocultar cronológicamente el tramo final.


## 7. Conclusión

Con más de dos semanas de antelación, mi predicción todavía se retrasaba más de cuatro días. El 29 de octubre el error se redujo prácticamente a cero, pero volvió a crecer en las últimas jornadas: con el último TLE disponible anticipó la caída unas ocho horas. La enseñanza no es que una fórmula sencilla pueda reemplazar a NASA, sino que incluso una trayectoria aparentemente estable puede cambiar muy deprisa cuando la atmósfera empieza a frenar de verdad al satélite.

## Fuentes y reproducción

- CelesTrak, archivo histórico TLE de Swift, NORAD 28485.
- NASA CCMC, *Swift Re-entry and Thermosphere Neutral Density Modeling Challenge*.
- NASA Science, confirmación de la reentrada del 4 de noviembre de 2026 a las 12:39 UTC.
- El notebook utiliza únicamente la biblioteca estándar de Python para facilitar su reproducción.
